# Rubik's Cube Diffusion Smoke Test

This notebook is the first bridge from our Rubik-specific arrangement operator into the **official Diffusion Illusions workflow**.

It is intentionally a smoke test, not the final illusion generator.

Its job is only to prove that we can:

1. load the official Diffusion Illusions code in Colab
2. load our Rubik arrangement spec and differentiable torch operator
3. create six learnable source faces
4. render Rubik views from those six faces
5. send those rendered views through the official Stable Diffusion `train_step`

This revision keeps the optimization focused on one target view:

- `solved:F`

That makes it easier to see whether stronger diffusion guidance can produce recognizable structure before we add multi-view competition back in.


## Before You Run

- Use a **GPU** runtime.
- The install cell may take a while.
- If this is your first time installing the official dependencies in this runtime, restart the runtime after the install cell finishes, then rerun from the repo bootstrap cell at the top.
- The official code may ask you to `huggingface-cli login` depending on the model/runtime state.

This notebook is trying to answer one question only:

**Can Stable Diffusion gradients see our Rubik views at all?**


In [ ]:
import os
import platform
import subprocess
import sys
from pathlib import Path

OUR_REPO_URL = 'https://github.com/netalondon/rubiks-diffusion-illusion.git'
OUR_REPO_DIR = Path('/content/rubiks-diffusion-illusion')
OFFICIAL_REPO_URL = 'https://github.com/RyannDaGreat/Diffusion-Illusions.git'
OFFICIAL_REPO_DIR = Path('/content/Diffusion-Illusions')

for repo_url, repo_dir in [
    (OUR_REPO_URL, OUR_REPO_DIR),
    (OFFICIAL_REPO_URL, OFFICIAL_REPO_DIR),
]:
    if repo_dir.exists():
        print(f'Reusing existing repo at {repo_dir}')
        subprocess.check_call(['git', '-C', str(repo_dir), 'pull', '--ff-only'])
    else:
        subprocess.check_call(['git', 'clone', repo_url, str(repo_dir)])

print('Python:', sys.version)
print('Platform:', platform.platform())
print('In Colab runtime:', 'COLAB_RELEASE_TAG' in os.environ)
print('Our repo:', OUR_REPO_DIR)
print('Official repo:', OFFICIAL_REPO_DIR)


In [ ]:
import subprocess
import sys

# The upstream notebook pins several older package versions that do not install cleanly on modern Colab runtimes.
# For this smoke test we only install the libraries the imported modules actually need.
third_party_packages = [
    'diffusers',
    'transformers',
    'rp',
    'easydict',
    'einops',
    'icecream',
    'imageio',
    'opencv-python',
    'pandas',
    'scipy',
    'tensorboard',
    'tensorboardX',
    'tqdm',
    'py3nvml',
]

subprocess.check_call([
    sys.executable,
    '-m',
    'pip',
    'install',
    '--upgrade',
    *third_party_packages,
])
subprocess.check_call([
    sys.executable,
    '-m',
    'pip',
    'install',
    '-r',
    str(OUR_REPO_DIR / 'requirements.txt'),
])

print('Installed a modern Colab-compatible subset of the Diffusion Illusions dependencies.')
print('If this was the first install in this runtime, restart the runtime now, then rerun from the repo bootstrap cell at the top.')


## Imports And Model Setup

The cells below mirror the official notebooks more closely:

- `source.stable_diffusion` comes from the official repo
- `LearnableImageFourier` comes from the official repo
- `render_all_arrangements_torch` comes from our repo


In [ ]:
import sys
from pathlib import Path

OUR_REPO_DIR = globals().get('OUR_REPO_DIR', Path('/content/rubiks-diffusion-illusion'))
OFFICIAL_REPO_DIR = globals().get('OFFICIAL_REPO_DIR', Path('/content/Diffusion-Illusions'))

for repo_dir in [OUR_REPO_DIR, OFFICIAL_REPO_DIR]:
    if not repo_dir.exists():
        raise FileNotFoundError(
            f'Missing expected repo directory: {repo_dir}. Run the repo bootstrap cell first.'
        )

if str(OUR_REPO_DIR) not in sys.path:
    sys.path.insert(0, str(OUR_REPO_DIR))
if str(OFFICIAL_REPO_DIR) not in sys.path:
    sys.path.insert(0, str(OFFICIAL_REPO_DIR))

import matplotlib.pyplot as plt
import numpy as np
import rp
import torch
import torch.nn as nn

import source.stable_diffusion as sd
from python_bridge import (
    FACE_FILE_NAMES,
    batch_to_pil_face_dict,
    clone_rendered_to_cpu,
    load_source_faces,
    load_spec,
    render_all_arrangements,
    render_all_arrangements_torch,
    save_rendered_faces,
    stack_source_face_tensors,
    tensor_to_pil,
)
from source.learnable_textures import LearnableImageFourier
from source.stable_diffusion_labels import NegativeLabel

gpu = rp.select_torch_device()
if str(gpu) == 'cpu':
    raise RuntimeError('This smoke test should run on a GPU runtime. Switch Colab to GPU and reconnect.')

if 's' not in globals():
    model_name = 'runwayml/stable-diffusion-v1-5'
    s = sd.StableDiffusion(gpu, model_name)

device = s.device
print('Stable Diffusion device:', device)


In [ ]:
SPEC_PATH = OUR_REPO_DIR / 'public' / 'generated' / 'rubiks-illusion-spec.json'
SOURCE_DIR = OUR_REPO_DIR / 'src' / 'assets' / 'face-art'

spec = load_spec(SPEC_PATH)
face_order = list(spec['primeImages'])
face_to_index = {face_id: index for index, face_id in enumerate(face_order)}

source_faces_pil = load_source_faces(SOURCE_DIR, face_order)
source_batch = stack_source_face_tensors(source_faces_pil, face_order, device=device)

pil_rendered = render_all_arrangements(spec, source_faces_pil)
torch_rendered = render_all_arrangements_torch(spec, source_batch, face_to_index)

max_diff = 0.0
for arrangement_name in pil_rendered:
    for face_id, pil_image in pil_rendered[arrangement_name].items():
        pil_tensor = stack_source_face_tensors({face_id: pil_image}, [face_id], device=device)[0]
        diff = (pil_tensor - torch_rendered[arrangement_name][face_id]).abs().max().item()
        max_diff = max(max_diff, diff)

print('Face order:', face_order)
print('Arrangement names:', list(spec['arrangements'].keys()))
print(f'Max abs difference between PIL and torch Rubik renders: {max_diff:.6f}')


In [ ]:
prompt_solved_f = 'a minimal graphic poster of a centered red circle on a warm cream background'
negative_prompt = 'blurry, noisy, muddy colors, text, watermark, low quality'

label_solved_f = NegativeLabel(prompt_solved_f, negative_prompt)

train_views = [
    {
        'name': 'solved:F',
        'arrangement': 'solved',
        'face': 'F',
        'prompt': prompt_solved_f,
        'label': label_solved_f,
        'weight': 1.0,
    },
]

for view in train_views:
    print(view['name'], '->', view['prompt'])


In [ ]:
class RubiksLearnableSourceFaces(nn.Module):
    def __init__(self, face_ids, size, stable_diffusion_device):
        super().__init__()
        self.face_ids = list(face_ids)
        self.generators = nn.ModuleDict(
            {
                face_id: LearnableImageFourier(
                    height=size,
                    width=size,
                    num_features=64,
                    hidden_dim=64,
                    scale=10,
                ).to(stable_diffusion_device)
                for face_id in self.face_ids
            }
        )

    def forward(self):
        return torch.stack([self.generators[face_id]() for face_id in self.face_ids], dim=0)


def show_face_dict(face_dict, title, face_ids):
    cols = min(3, max(1, len(face_ids)))
    rows = (len(face_ids) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(3.2 * cols, 3.2 * rows))
    axes = np.asarray(axes, dtype=object).reshape(rows, cols)
    fig.suptitle(title, fontsize=14)

    for axis in axes.flat:
        axis.axis('off')

    for index, face_id in enumerate(face_ids):
        row = index // cols
        col = index % cols
        axis = axes[row, col]
        axis.imshow(np.asarray(face_dict[face_id]))
        axis.set_title(face_id)
        axis.axis('off')

    plt.tight_layout()
    plt.show()


def show_training_views(rendered_views, title):
    items = [
        (
            view['name'],
            rendered_views[view['arrangement']][view['face']],
        )
        for view in train_views
    ]
    fig, axes = plt.subplots(1, len(items), figsize=(4.5 * len(items), 4.5))
    axes = np.asarray(axes, dtype=object).reshape(1, len(items))
    fig.suptitle(title, fontsize=14)

    for axis, (label, image_tensor) in zip(axes[0], items):
        axis.imshow(np.asarray(tensor_to_pil(image_tensor)))
        axis.set_title(label)
        axis.axis('off')

    plt.tight_layout()
    plt.show()


learnable_size = 96
learnable_faces = RubiksLearnableSourceFaces(face_order, learnable_size, device).to(device)


def get_current_state():
    current_source_batch = learnable_faces()
    current_rendered = render_all_arrangements_torch(spec, current_source_batch, face_to_index)
    return current_source_batch, current_rendered


initial_source_batch, initial_rendered = get_current_state()
show_face_dict(
    batch_to_pil_face_dict(initial_source_batch.detach().cpu(), face_order),
    'Initial learnable source faces',
    face_order,
)
show_training_views(clone_rendered_to_cpu(initial_rendered), 'Initial diffusion training views')


In [ ]:
NUM_ITER = 250
DISPLAY_INTERVAL = 25
LEARNING_RATE = 5e-5
GUIDANCE_SCALE = 80
NOISE_COEF = 0.12

s.max_step = 980
s.min_step = 20
optim = torch.optim.SGD(learnable_faces.parameters(), lr=LEARNING_RATE)

print('Running a stronger single-view smoke test.')
print('This is still not a final illusion notebook, but it should show whether one view can develop visible structure.')
print('Train views:', [view['name'] for view in train_views])
print('Iterations:', NUM_ITER)
print('Preview interval:', DISPLAY_INTERVAL)
print('Guidance scale:', GUIDANCE_SCALE)

for iter_num in range(NUM_ITER):
    current_source_batch, current_rendered = get_current_state()

    for view in train_views:
        current_view = current_rendered[view['arrangement']][view['face']][None]
        s.train_step(
            view['label'].embedding,
            current_view,
            noise_coef=NOISE_COEF * view['weight'],
            guidance_scale=GUIDANCE_SCALE,
        )

    optim.step()
    optim.zero_grad(set_to_none=True)

    if (iter_num + 1) % DISPLAY_INTERVAL == 0 or iter_num == 0:
        print(f'Completed iteration {iter_num + 1}/{NUM_ITER}')
        _, preview_rendered = get_current_state()
        show_training_views(clone_rendered_to_cpu(preview_rendered), f'Single-view preview at iter {iter_num + 1}')


In [ ]:
final_source_batch, final_rendered = get_current_state()
final_source_faces = batch_to_pil_face_dict(final_source_batch.detach().cpu(), face_order)
final_rendered_cpu = clone_rendered_to_cpu(final_rendered)

show_face_dict(final_source_faces, 'Final learnable source faces after single-view smoke test', face_order)
show_training_views(final_rendered_cpu, 'Final single-view training render')


In [ ]:
OUTPUT_ROOT = OUR_REPO_DIR / 'output' / 'colab-diffusion-smoke-test'
SOURCE_OUTPUT_DIR = OUTPUT_ROOT / 'source-faces'
RENDER_OUTPUT_DIR = OUTPUT_ROOT / 'rendered'

SOURCE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for face_id, image in final_source_faces.items():
    image.save(SOURCE_OUTPUT_DIR / f'{face_id}.png')

save_rendered_faces(
    {face_id: tensor_to_pil(image) for face_id, image in final_rendered_cpu['solved'].items()},
    RENDER_OUTPUT_DIR / 'solved',
)
save_rendered_faces(
    {face_id: tensor_to_pil(image) for face_id, image in final_rendered_cpu['scrambled'].items()},
    RENDER_OUTPUT_DIR / 'scrambled',
)

print('Saved source faces to:', SOURCE_OUTPUT_DIR)
print('Saved rendered solved faces to:', RENDER_OUTPUT_DIR / 'solved')
print('Saved rendered scrambled faces to:', RENDER_OUTPUT_DIR / 'scrambled')


## What Success Looks Like

This notebook is a success if all of these are true:

- Stable Diffusion loads successfully
- the longer single-view loop runs without shape/device errors
- the rendered `solved:F` view visibly moves away from noise over time
- outputs are saved under `output/colab-diffusion-smoke-test`

Once that works, the next step is to compare this single-view result against a multi-view version and decide how much extra structure survives when we add competing constraints back in.
